# Preliminary BACE Benchmark

Minimal reproduction of the BACE control-chain figure: PyG ChebNet (external baseline,
matched to TAME's own tuned hyperparameters), TAME/TAME-Fusion graph-only ablations, and
the full TAME/TAME-Fusion models, over 100 seeds on the scaffold split.

Data: `data/bace_preliminary_results.csv` (produced by `scripts/bace_preliminary_results.py`)
and `data/bace_pyg_chebnet_tame_hps_results.csv` (produced by
`scripts/pyg_chebnet_bace_matched_tame.py`).

In [ ]:
import json
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.dpi"] = 140

results_dir = Path("data")

In [ ]:
df = pd.read_csv(results_dir / "bace_preliminary_results.csv")
with open(results_dir / "bace_preliminary_meta.json") as f:
    meta = json.load(f)

pyg_csvs = sorted(results_dir.glob("bace_pyg_chebnet*results.csv"))
if pyg_csvs:
    df_pyg = pd.concat([pd.read_csv(p) for p in pyg_csvs], ignore_index=True)
    df = pd.concat([df_pyg, df], ignore_index=True)

ORDER = ["PyG ChebNet (TAME HPs)", "TAME (graph-only)", "TAME-Fusion (graph-only)",
         "TAME", "TAME-Fusion"]
models = [m for m in ORDER if m in set(df["Model"])]
df["Model"] = pd.Categorical(df["Model"], categories=models, ordered=True)
df = df.sort_values(["Model", "Seed"]).reset_index(drop=True)

METRICS = ["Test_ROC_AUC", "Test_PR_AUC", "Test_Macro_F1"]
METRIC_LABELS = {"Test_ROC_AUC": "ROC-AUC", "Test_PR_AUC": "PR-AUC", "Test_Macro_F1": "Macro-F1"}

n_seeds = df.groupby("Model", observed=True)["Seed"].nunique().max()
print(f"Split: {meta['split_sizes']}  |  seeds: {n_seeds}")
print(df.groupby("Model", observed=True)[METRICS].agg(["mean", "std"]).round(4).to_string())

In [ ]:
palette = {
    "PyG ChebNet (TAME HPs)": "#f97316",
    "TAME (graph-only)": "#93c5fd",
    "TAME-Fusion (graph-only)": "#6ee7b7",
    "TAME": "#3b82f6",
    "TAME-Fusion": "#10b981",
}
palette = {m: palette[m] for m in models}

legend_handles = [plt.Rectangle((0, 0), 1, 1, facecolor=palette[m], edgecolor="black", linewidth=0.8)
                   for m in models]


def _style_ax(ax, metric):
    ax.set_title(METRIC_LABELS[metric], fontsize=17, fontweight="bold")
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.set_xticklabels([])
    ax.tick_params(axis="x", length=0)
    ax.spines[["top", "right"]].set_visible(False)


def _box_strip(ax, metric):
    sns.boxplot(data=df, x="Model", y=metric, order=models, hue="Model",
                palette=palette, width=0.6, fliersize=0, legend=False, ax=ax)
    sns.stripplot(data=df, x="Model", y=metric, order=models,
                  color="black", alpha=0.35, size=4, jitter=0.15, ax=ax)
    _style_ax(ax, metric)


# Figure 1: ROC-AUC + Macro-F1, side by side 1x2
fig1, axes1 = plt.subplots(1, 2, figsize=(16, 6))
for ax, metric in zip(axes1, ["Test_ROC_AUC", "Test_Macro_F1"]):
    _box_strip(ax, metric)
fig1.legend(legend_handles, models, loc="lower center", ncol=len(models),
            frameon=False, bbox_to_anchor=(0.5, -0.08), fontsize=13)
fig1.tight_layout()
plt.show()

# Figure 2: PR-AUC alone
fig2, ax2 = plt.subplots(1, 1, figsize=(8, 6))
_box_strip(ax2, "Test_PR_AUC")
fig2.legend(legend_handles, models, loc="lower center", ncol=len(models),
            frameon=False, bbox_to_anchor=(0.5, -0.08), fontsize=13)
fig2.tight_layout()
plt.show()